In [9]:
#Imports
import torch
import torch.nn as nn

In [10]:
#Test Data
t = torch.linspace(0, 100, 500)
series = torch.sin(t)

p = 10

X, Y = [], []
for i in range(len(series) - p):
    X.append(series[i:i+p])
    Y.append(series[i+p])

X = torch.stack(X)
Y = torch.stack(Y).unsqueeze(1)

X = X.unsqueeze(2)             
print("X shape:", X.shape)       
print("Y shape:", Y.shape)     

X shape: torch.Size([490, 10, 1])
Y shape: torch.Size([490, 1])


In [11]:
#LSTM
class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size = 1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first = True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        output, (hidden_state, cell_state) = self.lstm(x)
        last_hidden_state = hidden_state[-1]
        y_hat = self.output(last_hidden_state)
        return y_hat

model = LSTM(input_size = 1, hidden_size = 50, num_layers = 1)
print(model)

LSTM(
  (lstm): LSTM(1, 50, batch_first=True)
  (output): Linear(in_features=50, out_features=1, bias=True)
)


In [12]:
#Loss and Optimizer
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

In [14]:
#Training Loop
num_epochs = 300
for epoch in range(num_epochs):
    prediction = model(X)

    loss = loss_function(prediction, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 30 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')

Epoch 0, Loss: 0.5031358599662781
Epoch 30, Loss: 0.006059648934751749
Epoch 60, Loss: 0.0007720752619206905
Epoch 90, Loss: 0.00023959034297149628
Epoch 120, Loss: 8.619638538220897e-05
Epoch 150, Loss: 2.36892174143577e-05
Epoch 180, Loss: 6.588811174879083e-06
Epoch 210, Loss: 3.614913339333725e-06
Epoch 240, Loss: 2.6689715468819486e-06
Epoch 270, Loss: 2.007138618864701e-06


In [15]:
#Evaluation
def evaluate(model, X, Y):
    model.eval
    with torch.no_grad():
        predictions = model(X)
        return loss_function(predictions, Y).item()

print(f"\nfinal MSE: {evaluate(model, X, Y):.6f}")


final MSE: 0.000002
